# Creation of the Data Base

In [1]:
import duckdb
import numpy
import pandas
import os

In [2]:
parent_path = "C:/Users/quintyn_come/Codex/AIBT102/"

## Data Base Initializaton

In [3]:
database_folder = "database"
database_name = "database_duckdb"
cleaned_data_path = "cleaning/cleaned_data"

In [4]:
database_path = os.path.join(parent_path, database_folder, database_name)

In [5]:
if os.path.exists(database_path):
    os.remove(database_path)
con = duckdb.connect(database_path)

## Geographical Area

In [6]:
geographical_area_name = "geographical_area.csv"

In [7]:
geographical_area_path = os.path.join(parent_path, cleaned_data_path, geographical_area_name)

In [8]:
con.execute("""
    CREATE OR REPLACE TABLE GEOGRAPHICAL_AREA (
        IRIS VARCHAR PRIMARY KEY,
        commune_name VARCHAR,
        iris_name VARCHAR,
        surface_m2 DOUBLE,
    )
""")
con.execute("""
    INSERT INTO GEOGRAPHICAL_AREA
    SELECT * FROM read_csv_auto(?)
""", [geographical_area_path])
print(con.execute(f"DESCRIBE GEOGRAPHICAL_AREA").fetchdf())

    column_name column_type null  key default extra
0          IRIS     VARCHAR   NO  PRI    None  None
1  commune_name     VARCHAR  YES  NaN    None  None
2     iris_name     VARCHAR  YES  NaN    None  None
3    surface_m2      DOUBLE  YES  NaN    None  None


In [9]:
con.execute("""
    SELECT * FROM GEOGRAPHICAL_AREA
    LIMIT 1
""").fetchdf()

,IRIS,commune_name,iris_name,surface_m2
0,670430101,Bischheim,Annexe,1.623617e+06


## Income Statistics

In [10]:
income_statistics_name = "income_statistics.csv"

In [11]:
income_statistics_path = os.path.join(parent_path, cleaned_data_path, income_statistics_name)

In [12]:
con.execute("""
    CREATE OR REPLACE TABLE INCOME_STATISTICS (
        income_statistics_id VARCHAR PRIMARY KEY,
        IRIS VARCHAR,
        poverty_rate DOUBLE,
        median_income DOUBLE,
        FOREIGN KEY (IRIS) REFERENCES GEOGRAPHICAL_AREA(IRIS),
    )
""")
con.execute("""
    INSERT INTO INCOME_STATISTICS
    SELECT * FROM read_csv_auto(?)
""", [income_statistics_path])
print(con.execute(f"DESCRIBE INCOME_STATISTICS").fetchdf())

            column_name column_type null  key default extra
0  income_statistics_id     VARCHAR   NO  PRI    None  None
1                  IRIS     VARCHAR  YES  NaN    None  None
2          poverty_rate      DOUBLE  YES  NaN    None  None
3         median_income      DOUBLE  YES  NaN    None  None


In [13]:
con.execute("""
    SELECT * FROM INCOME_STATISTICS
    LIMIT 1
""").fetchdf()

,income_statistics_id,IRIS,poverty_rate,median_income
0,1,010040101,19.024849,20343.69273


## Population Statistics

In [14]:
population_statistics_name = "population_statistics.csv"

In [15]:
population_statistics_path = os.path.join(parent_path, cleaned_data_path, population_statistics_name)

In [16]:
con.execute("""
    CREATE OR REPLACE TABLE POPULATION_STATISTICS (
        population_statistics_id VARCHAR PRIMARY KEY,
        IRIS VARCHAR,
        population INT,
        FOREIGN KEY (IRIS) REFERENCES GEOGRAPHICAL_AREA(IRIS),
    )
""")
con.execute("""
    INSERT INTO POPULATION_STATISTICS
    SELECT
        p.population_statistics_id,
        p.IRIS,
        p.population
    FROM read_csv_auto(?) AS p
    INNER JOIN GEOGRAPHICAL_AREA AS g
        ON p.IRIS = g.IRIS
""", [population_statistics_path]) # We can have some IRIS that aren't registered in the GEOGRAPHICAL AREA but if we were using all the data we wouldn't have any issues, if with all the data there are still issues, it's probably because it's some data from outside metropolitan France
print(con.execute(f"DESCRIBE POPULATION_STATISTICS").fetchdf())

                column_name column_type null  key default extra
0  population_statistics_id     VARCHAR   NO  PRI    None  None
1                      IRIS     VARCHAR  YES  NaN    None  None
2                population     INTEGER  YES  NaN    None  None


In [17]:
con.execute("""
    SELECT * FROM POPULATION_STATISTICS
    LIMIT 1
""").fetchdf()

,population_statistics_id,IRIS,population
0,8392,670430101,51


Let's quickly check the number or data we have:

In [18]:
number_of_rows_population_statistics = con.execute("""
    SELECT COUNT(*) AS number_of_rows
    FROM POPULATION_STATISTICS
""").fetchone()[0]
print("Number of rows in population_statistics: ", number_of_rows_population_statistics)

Number of rows in population_statistics:  15571


In [19]:
number_of_rows_geographical_area = con.execute("""
    SELECT COUNT(*) AS number_of_rows
    FROM GEOGRAPHICAL_AREA
""").fetchone()[0]
print("Number of rows in geographical_area: ", number_of_rows_geographical_area)

Number of rows in geographical_area:  48512


It seems that we have the population data only on 1/3 of the rows.

Let's check the total number of IRIS for whose we have both income and population data:

In [20]:
coverage_rate = con.execute("""
    SELECT
        COUNT(DISTINCT g.IRIS) AS total_iris,
        COUNT(DISTINCT CASE
            WHEN p.IRIS IS NOT NULL AND i.IRIS IS NOT NULL
            THEN g.IRIS
        END) AS valid_iris,
        ROUND(
            100.0 * COUNT(DISTINCT CASE
                WHEN p.IRIS IS NOT NULL AND i.IRIS IS NOT NULL
                THEN g.IRIS
            END) / COUNT(DISTINCT g.IRIS),
            2
        ) AS coverage_percentage
    FROM GEOGRAPHICAL_AREA AS g
    LEFT JOIN POPULATION_STATISTICS AS p
        ON g.IRIS = p.IRIS
    LEFT JOIN INCOME_STATISTICS AS i
        ON g.IRIS = i.IRIS
""").fetchdf()

print(coverage_rate)

   total_iris  valid_iris  coverage_percentage
0       48512       15496                31.94


It seems that the ratio doesn't change much.
It is important to notice that there are only about 15.5k valid IRIS in France, zome zones haven't been attributed a proper IRIS because they don't have enough inhabitants.
Therefore there is no problems with the data.

## Business

In [21]:
business_name = "businesses.csv"

In [22]:
business_path = os.path.join(parent_path, cleaned_data_path, business_name)

In [23]:
con.execute("""
    CREATE OR REPLACE TABLE BUSINESS (
        osm_id VARCHAR PRIMARY KEY,
        IRIS VARCHAR,
        name VARCHAR,
        latitude DOUBLE,
        longitude DOUBLE,
        FOREIGN KEY (IRIS) REFERENCES GEOGRAPHICAL_AREA(IRIS),
    )
""")
con.execute("""
    INSERT INTO BUSINESS
    SELECT
        b.osm_id,
        b.IRIS,
        b.name,
        b.latitude,
        b.longitude
    FROM read_csv_auto(
        ?,
        types = {'IRIS': 'VARCHAR'}
    ) AS b
    INNER JOIN GEOGRAPHICAL_AREA AS g
        ON b.IRIS = g.IRIS
""", [business_path])
print(con.execute(f"DESCRIBE BUSINESS").fetchdf())

  column_name column_type null  key default extra
0      osm_id     VARCHAR   NO  PRI    None  None
1        IRIS     VARCHAR  YES  NaN    None  None
2        name     VARCHAR  YES  NaN    None  None
3    latitude      DOUBLE  YES  NaN    None  None
4   longitude      DOUBLE  YES  NaN    None  None


In [24]:
con.execute("""
    SELECT * FROM BUSINESS
    LIMIT 1
""").fetchdf()

,osm_id,IRIS,name,latitude,longitude
0,25213384,751156001,Relais Total Du Pont Aval,48.834677,2.266146


## Business Category

In [25]:
business_category_name = "categories.csv"

In [26]:
business_category_path = os.path.join(parent_path, cleaned_data_path, business_category_name)

In [27]:
con.execute("""
    CREATE OR REPLACE TABLE BUSINESS_CATEGORY (
        category_id VARCHAR PRIMARY KEY,
        category VARCHAR,
        category_type VARCHAR,
    )
""")
con.execute("""
    INSERT INTO BUSINESS_CATEGORY
    SELECT * FROM read_csv_auto(?)
""", [business_category_path])
print(con.execute(f"DESCRIBE BUSINESS_CATEGORY").fetchdf())

     column_name column_type null  key default extra
0    category_id     VARCHAR   NO  PRI    None  None
1       category     VARCHAR  YES  NaN    None  None
2  category_type     VARCHAR  YES  NaN    None  None


In [28]:
con.execute("""
    SELECT * FROM BUSINESS_CATEGORY
    LIMIT 1
""").fetchdf()

,category_id,category,category_type
0,1,Aide à la personne,office


## Belongs

In [29]:
belongs_name = "category_business_link.csv"

In [30]:
belongs_path = os.path.join(parent_path, cleaned_data_path, belongs_name)

In [31]:
con.execute("""
    CREATE OR REPLACE TABLE BELONGS (
        osm_id VARCHAR,
        category_id VARCHAR,
        PRIMARY KEY (osm_id, category_id),
        FOREIGN KEY (osm_id) REFERENCES BUSINESS(osm_id),
        FOREIGN KEY (category_id) REFERENCES BUSINESS_CATEGORY(category_id)
    )
""")
con.execute("""
    INSERT INTO BELONGS
    SELECT * FROM read_csv_auto(?)
""", [belongs_path])
print(con.execute(f"DESCRIBE BELONGS").fetchdf())

   column_name column_type null  key default extra
0       osm_id     VARCHAR   NO  PRI    None  None
1  category_id     VARCHAR   NO  PRI    None  None


In [32]:
con.execute("""
    SELECT * FROM BELONGS
    LIMIT 1
""").fetchdf()

,osm_id,category_id
0,25213384,127


# Creation of the Data Marts

## Dim Geography

## Dim Business Category

## Fact Business Indicators

## Fact Business Indicators